### NumPy Exercises: Supermarket Sales

#### 📋 Dataset Information

**Dataset**: Supermarket Sales  
**Source**: Attached file  

**Columns**:
- `Branch`: Store branch identifier where the transaction occurred ('A', 'B', or 'C').
- `Customer type`: Customer membership status ('Member', 'Normal').
- `Gende`r: Customer's gender ('Male' or 'Female').
- `Product line`: Product category purchased (6 categories: Electronic accessories, Fashion accessories, Food and beverages, Health and beauty, Home and lifestyle, Sports and travel).
- `Quantity`: Number of items purchased in the transaction.
- `Total`: Total amount paid by customer in dollars, including tax.
- `Date`: Transaction date in M/D/YYYY format.
- `Rating`: Customer satisfaction rating on a scale of 0.0 to 10.0 (higher is better).

In [1]:
import numpy as np

In [2]:
## NO NEED CODE HERE
# Load the CSV file
data = np.genfromtxt(
    'Data/supermarket_sales.csv',
    delimiter=',',
    skip_header=1,    # Skip header row
    dtype=str         # Load as strings (no preprocessing)
)

# Print basic info
print("="*60)
print("DATASET LOADED")
print("="*60)
print(f"Shape: {data.shape}")
print(f"  Rows: {data.shape[0]} transactions")
print(f"  Columns: {data.shape[1]}")

# Column names
columns = ['Branch', 'Customer type', 'Gender', 'Product line', 
           'Quantity', 'Total', 'Date', 'Rating']

print(f"\nColumns:")
for i, col in enumerate(columns):
    print(f"  [{i}] {col}")

DATASET LOADED
Shape: (1000, 8)
  Rows: 1000 transactions
  Columns: 8

Columns:
  [0] Branch
  [1] Customer type
  [2] Gender
  [3] Product line
  [4] Quantity
  [5] Total
  [6] Date
  [7] Rating


In [3]:
# Check data types and convert where necessary
print(f"Overall Array Dtype: {data.dtype}")

for i, col in enumerate(columns):
    sample_val = data[0, i]
    val_type = type(sample_val).__name__
    
    print(f" [{i}] | {col:<20} | {val_type:<15} | {sample_val}")


branch_col  = data[:, 0]
type_col    = data[:, 1]
gender_col  = data[:, 2]
product_col = data[:, 3]

quantity_col = data[:, 4].astype(float)
total_col    = data[:, 5].astype(float)
rating_col   = data[:, 7].astype(float)


Overall Array Dtype: <U22
 [0] | Branch               | str_            | A
 [1] | Customer type        | str_            | Member
 [2] | Gender               | str_            | Female
 [3] | Product line         | str_            | Health and beauty
 [4] | Quantity             | str_            | 7
 [5] | Total                | str_            | 548.9715
 [6] | Date                 | str_            | 1/5/2019
 [7] | Rating               | str_            | 9.1


### Task 1: Customer Segmentation Analysis
**Which customer segment (by type and gender) generates the highest revenue per transaction, and how does their rating behavior differ from other segments?** 

**Instruction:**
- 4 customer segments: Member-Female, Member-Male, Normal-Female, Normal-Male.
- Calculate revenue per transaction for each segment.
- Calculate average rating for each segment.
- How each segment compares to the overall average?
   - Calculate deviations 
- Which segment is the "best"? (Based on revenue, rating and count)
   - Weight: 50% revenue, 30% rating, 20% count (volume) (Need normalize to [0.1])

In [4]:
# Create key for segment
keys = np.char.add(type_col, "-")
keys = np.char.add(keys, gender_col)

unique_segments, seg_idx = np.unique(keys, return_inverse=True)

seg_counts = np.bincount(seg_idx)
seg_rev_sum = np.bincount(seg_idx, weights=total_col)
seg_rating_sum = np.bincount(seg_idx, weights=rating_col)

seg_avg_rev = seg_rev_sum / seg_counts
seg_avg_rating = seg_rating_sum / seg_counts

ov_rev = np.mean(total_col)
ov_rating = np.mean(rating_col)

# Cal Percentage Difference
# (Segment - Overall) / Overall * 100
rev_diff_pcts = ((seg_avg_rev - ov_rev) / ov_rev) * 100
rate_diff_pcts = ((seg_avg_rating - ov_rating) / ov_rating) * 100


def normalize(arr):
    return (arr - np.min(arr)) / (np.max(arr) - np.min(arr))

norm_rev = normalize(seg_avg_rev)
norm_rate = normalize(seg_avg_rating)
norm_count = normalize(seg_counts)

# Cal final scores
scores = (0.5 * norm_rev) + (0.3 * norm_rate) + (0.2 * norm_count)
best_idx = np.argmax(scores)

print("Overall Averages:")
print(f"    Revenue per transaction: ${ov_rev:.3f}")
print(f"    Rating: {ov_rating:.3f}")

print("\nSegment Performance:")

for i, seg in enumerate(unique_segments):
    print(f"\n{seg}:")
    
    # Count stats
    count = seg_counts[i]
    percen_count = (count / len(total_col)) * 100
    print(f"  Count:  {count} transactions ({percen_count:.3f}%)")
    
    # Revenue stats
    rev = seg_avg_rev[i]
    rev_diff = rev_diff_pcts[i]
    rev_sign = "+" if rev_diff >= 0 else ""
    print(f"  Avg Revenue: ${rev:.3f} ({rev_sign}{rev_diff:.3f}% vs overall)")
    
    # Rating stats
    rating = seg_avg_rating[i]
    rating_diff = rate_diff_pcts[i]
    rating_sign = "+" if rating_diff >= 0 else ""
    print(f"  Avg Rating: {rating:.3f} ({rating_sign}{rating_diff:.3f}% vs overall)")

print("\nSegment Ranking:")

sorted_indices = np.argsort(scores)[::-1]

for rank, idx in enumerate(sorted_indices, 1):
    print(f"{rank}. {unique_segments[idx]:<14} - Score: {scores[idx]:.3f}")

print(f"\nBest Segment: {unique_segments[best_idx]}")

Overall Averages:
    Revenue per transaction: $322.967
    Rating: 6.973

Segment Performance:

Member-Female:
  Count:  261 transactions (26.100%)
  Avg Revenue: $337.728 (+4.570% vs overall)
  Avg Rating: 6.941 (-0.460% vs overall)

Member-Male:
  Count:  240 transactions (24.000%)
  Avg Revenue: $316.985 (-1.852% vs overall)
  Avg Rating: 6.940 (-0.469% vs overall)

Normal-Female:
  Count:  240 transactions (24.000%)
  Avg Revenue: $332.233 (+2.869% vs overall)
  Avg Rating: 6.990 (+0.254% vs overall)

Normal-Male:
  Count:  259 transactions (25.900%)
  Avg Revenue: $305.048 (-5.548% vs overall)
  Avg Rating: 7.019 (+0.663% vs overall)

Segment Ranking:
1. Member-Female  - Score: 0.702
2. Normal-Female  - Score: 0.608
3. Normal-Male    - Score: 0.481
4. Member-Male    - Score: 0.183

Best Segment: Member-Female


### Task 2: Product Performance Optimization
**Which product lines are underperforming (below average sales) in which branches, and what is the revenue opportunity if they reached branch-average performance?**

**Instructions**:
1. **18 combinations**: 3 Branches × 6 Product Lines = 18
2. **Average sales per product-branch combination**
3. **Identify underperformers**: Which are below their branch average?
4. **Calculate gap**: How much below average?
5. **Revenue opportunity**: Potential gain if they reached branch average

In [5]:
print(f"Branches: {', '.join(np.unique(branch_col))}")
unique_products = np.unique(product_col)
print(f"Product Lines: {len(unique_products)} categories")

# Cal Branch Average
unique_branches, branch_idx = np.unique(branch_col, return_inverse=True)
branches_counts = np.bincount(branch_idx)
branches_sums = np.bincount(branch_idx, weights=total_col)
branches_avgs = branches_sums / branches_counts

# Cal Branch-Product Average
keys = np.char.add(branch_col, "|") 
keys = np.char.add(keys, product_col)

u_bp, bp_idx = np.unique(keys, return_inverse=True)
bp_counts = np.bincount(bp_idx)
bp_avgs = np.bincount(bp_idx, weights=total_col) / bp_counts

# Helper dict for lookup
# Key: "Branch|Product", Value: (Avg, Count)
perf_map = {k: (a, c) for k, a, c in zip(u_bp, bp_avgs, bp_counts)}

grand_total_opp = 0

for i, branch in enumerate(unique_branches):
    branch_avg = branches_avgs[i]
    print(f"Branch {branch} (Average: ${branch_avg:.3f}):")
    
    branch_opp = 0
    
    for prod in unique_products:
        key = f"{branch}|{prod}"
        
        if key in perf_map:
            actual_avg, count = perf_map[key]
            
            gap = branch_avg - actual_avg
            
            if gap > 0:
                opp = gap * count
                branch_opp += opp     
                print(f"  X  {prod:<25} : $ {actual_avg:7.3f} (${gap:6.3f} below avg, ${opp:8.3f} opportunity)")
            else:
                gap_abs = abs(gap)
                print(f"  O  {prod:<25} : $ {actual_avg:7.3f} (${gap_abs:6.3f} above avg, {'-':>9} opportunity)")
        else:
            print(f"    {prod:<25} : No Sales")

    grand_total_opp += branch_opp
    print(f"\n  Branch {branch} Opportunity: ${branch_opp:,.3f}\n")
    print()

print(f"Total revenue opportunity: ${grand_total_opp:,.3f}")


Branches: A, B, C
Product Lines: 6 categories
Branch A (Average: $312.354):
  X  Electronic accessories    : $ 305.285 ($ 7.069 below avg, $ 424.128 opportunity)
  O  Fashion accessories       : $ 320.245 ($ 7.891 above avg,         - opportunity)
  X  Food and beverages        : $ 295.916 ($16.439 below avg, $ 953.433 opportunity)
  X  Health and beauty         : $ 268.037 ($44.317 below avg, $2082.886 opportunity)
  O  Home and lifestyle        : $ 344.880 ($32.526 above avg,         - opportunity)
  O  Sports and travel         : $ 328.351 ($15.997 above avg,         - opportunity)

  Branch A Opportunity: $3,460.448


Branch B (Average: $319.873):
  X  Electronic accessories    : $ 310.026 ($ 9.846 below avg, $ 541.544 opportunity)
  X  Fashion accessories       : $ 264.731 ($55.142 below avg, $3418.779 opportunity)
  X  Food and beverages        : $ 304.298 ($15.575 below avg, $ 778.737 opportunity)
  O  Health and beauty         : $ 376.994 ($57.121 above avg,         - opportuni

### Task 3: High-Value Customer Identification
**What percentage of total revenue comes from the top 20% of transactions, and what are the common characteristics of these high-value transactions?**

**Instructions**:\
**1. Identify the Top 20% of Transactions by Total amount**

**2. Calculate Revenue Concentration**\
Determine what percentage of total revenue is generated by these top 20% of transactions.

**3. Profile High-Value Transaction Characteristics**\
For the top 20% transactions, analyze their common patterns across multiple dimensions. How many transaction happen: 
 - On each branches
 - On each product lines
 - On each customer types
 - On each gender

**4. Compare High-Value vs Overall Distribution**\
Calculate how the characteristics of high-value transactions differ from the overall dataset by comparing percentage distributions across branches, products, customer types, and gender. \
For example, if Branch A represents 40% of high-value transactions but only 33% of all transactions, this indicates Branch A has a premium customer base. 

**5. Calculate Average Metrics for High-Value Segment**\
Compute the average transaction amount, average quantity purchased, and average rating specifically for the top 20% group and compare these averages to the overall dataset averages. For example:
- Top 20% spend 125% MORE per transaction
- Top 20% buy 55% MORE items per transaction
- Top 20% Ratings ....
  
**6. Identify the "Golden Combination"**\
Find the most common combination of characteristics (Branch + Product + Customer Type + Gender) within the high-value segment

**7. Analyze Contribution by Percentile Groups**\
Beyond just the top 20%, how revenue is distributed across all percentile groups (top 20%, 20-40%, 40-60%, 60-80%, bottom 20%) 

In [6]:
sorted_indices = np.argsort(total_col)[::-1]
n_rows = len(total_col)
top_20_count = int(n_rows * 0.2)

top_20_idx = sorted_indices[:top_20_count] # Index of Top 20%

top_total = total_col[top_20_idx]
top_branch = branch_col[top_20_idx]
top_prod = product_col[top_20_idx]
top_type = type_col[top_20_idx]
top_gender = gender_col[top_20_idx]
top_qty = quantity_col[top_20_idx]
top_rate = rating_col[top_20_idx]

total_revenue_sum = np.sum(total_col)

print("Revenue Concentration:")
rev_top = np.sum(top_total)
rev_share = (rev_top / total_revenue_sum) * 100
print(f"    Top 20% transactions: {top_20_count} out of {n_rows}")
print(f"    Revenue from top 20%: ${rev_top:.2f} ({rev_share:.1f}% of total)")

print("\nHigh-Value Transaction Profile:") 
        
def print_distribution(label, top_arr, all_arr):
    print(f"\n  {label}:")
    uniques = np.unique(all_arr)
    for u in uniques:
        # Cal count in Top 20%
        count_in_top = np.sum(top_arr == u) 
        # Cal % in Top 20%
        top_pct = np.mean(top_arr == u) * 100
        
        # Cal % in Overall
        all_pct = np.mean(all_arr == u) * 100
        # Cal diff
        diff = top_pct - all_pct
        sign = "+" if diff >= 0 else "-"
        
        print(f"    {u:<25}: {count_in_top:3d} transactions ({top_pct:4.1f}%) | Overall: {all_pct:4.1f}% ({sign}{abs(diff):.1f}% diff)")

print_distribution("Branch Distribution", top_branch, branch_col)
print_distribution("Product Line Distribution", top_prod, product_col)
print_distribution("Customer Type", top_type, type_col)
print_distribution("Gender", top_gender, gender_col)


print("\nAverage Metrics Comparison:")

# Helper function to print comparison 
def print_metric_cmp(name, each_val, overall_val, is_money=False, is_pct_diff=True):
    prefix = "$" if is_money else ""
    
    top_str = f"{prefix}{each_val:.2f}"
    all_str = f"{prefix}{overall_val:.2f}"
    
    if is_pct_diff:
        diff = ((each_val - overall_val) / overall_val) * 100
        note = f"({diff:+.1f}% MORE)" if diff > 0 else f"({diff:.1f}% LESS)"
    else:
        diff = each_val - overall_val
        note = f"({diff:+.1f}%)"

    print(f"    {name:<20}: {top_str:>9} vs {all_str:<8} - Overall {note}")

print_metric_cmp("Transaction amount", np.mean(top_total), np.mean(total_col), is_money=True)
print_metric_cmp("Quantity purchased", np.mean(top_qty), np.mean(quantity_col))
print_metric_cmp("Rating", np.mean(top_rate), np.mean(rating_col))


print("\nMost Common 'Golden Combination' within the high-value segment (Top 20%):")
# Create key for combination
combo_keys = np.array([f"{b}|{p}|{t}|{g}" for b, p, t, g in zip(top_branch, top_prod, top_type, top_gender)])
# Find mode
u_combos, counts = np.unique(combo_keys, return_counts=True)
best_idx = np.argmax(counts)
best_combo = u_combos[best_idx]
best_count = counts[best_idx]
b, p, t, g = best_combo.split("|")

print(f"  Branch: {b}")
print(f"  Product: {p}")
print(f"  Customer Type: {t}")
print(f"  Gender: {g}")
print(f"  Occurrences: {best_count} times")


print("\nRevenue Distribution by Percentile Groups:")


chunks = np.array_split(sorted_indices, 5)
labels = ["Top 20%", "20-40%", "40-60%", "60-80%", "Bottom 20%"]

for i, label in enumerate(labels):
    idx_chunk = chunks[i]
    rev_chunk = total_col[idx_chunk]
    
    sum_val = np.sum(rev_chunk)
    share_val = (sum_val / total_revenue_sum) * 100
    avg_val = np.mean(rev_chunk)
    
    print(f"  {label:<12}: $ {sum_val:9.2f} ( {share_val:4.1f}% of total, Avg: ${avg_val:6.2f})")
    

Revenue Concentration:
    Top 20% transactions: 200 out of 1000
    Revenue from top 20%: $145596.14 (45.1% of total)

High-Value Transaction Profile:

  Branch Distribution:
    A                        :  60 transactions (30.0%) | Overall: 34.0% (-4.0% diff)
    B                        :  67 transactions (33.5%) | Overall: 33.2% (+0.3% diff)
    C                        :  73 transactions (36.5%) | Overall: 32.8% (+3.7% diff)

  Product Line Distribution:
    Electronic accessories   :  37 transactions (18.5%) | Overall: 17.0% (+1.5% diff)
    Fashion accessories      :  31 transactions (15.5%) | Overall: 17.8% (-2.3% diff)
    Food and beverages       :  31 transactions (15.5%) | Overall: 17.4% (-1.9% diff)
    Health and beauty        :  32 transactions (16.0%) | Overall: 15.2% (+0.8% diff)
    Home and lifestyle       :  33 transactions (16.5%) | Overall: 16.0% (+0.5% diff)
    Sports and travel        :  36 transactions (18.0%) | Overall: 16.6% (+1.4% diff)

  Customer Type:
  